In [ ]:
import torch
import pandas as pd
from datasets import Dataset
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments,AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import train_test_split
from transformers import TrainingArguments, Trainer


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv("/content/news.csv")

dataset = Dataset.from_pandas(df)

split_data = dataset.train_test_split(
    test_size=0.05,
    seed=42
)

train_ds = split_data["train"]
test_ds  = split_data["test"]

print("Train size:", len(train_ds))
print("Test size:", len(test_ds))


In [ ]:
model_name = "google/mt5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
MAX_INPUT = 512
MAX_TARGET = 64

def preprocess(batch):
    inputs = ["headline: " + x for x in batch["text"]]

    model_inputs = tokenizer(
        inputs,
        truncation=True,
        padding="max_length",
        max_length=MAX_INPUT
    )

    labels = tokenizer(
        batch["headline"],
        truncation=True,
        padding="max_length",
        max_length=MAX_TARGET
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
test_tok  = test_ds.map(preprocess,  batched=True, remove_columns=test_ds.column_names)


In [ ]:
args = TrainingArguments(
    output_dir="/content/mt5_burmese_headline",
    evaluation_strategy="steps",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    learning_rate=2e-4,
    fp16=True,
    save_steps=1000,
    logging_steps=100,
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok
)

trainer.train()


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

text = "သတင်းအပြည့်အစုံကို ဒီမှာထည့်ပါ..."

inputs = tokenizer(
    "headline: " + text,
    return_tensors="pt",
    truncation=True
).to(device)

out = model.generate(
    **inputs,
    max_length=64,
    num_beams=4,
    early_stopping=True
)

print("Generated headline:")
print(tokenizer.decode(out[0], skip_special_tokens=True))


In [ ]:
trainer.save_model("/content/burmese_headline_model")
tokenizer.save_pretrained("/content/burmese_headline_model")
